# WEEK 6 - DAY 4: FINAL INTERACTIVE SALES DASHBOARD

## Objective

The objective of Day 4 is to combine the data analysis and visualization techniques developed during Week 6 into a professional interactive sales dashboard.

## Dashboard Features

- Key Performance Indicators (KPIs)
- Daily sales trend analysis
- Product-wise sales analysis
- Region-wise sales analysis
- Quantity versus total sales analysis
- Interactive product filtering
- Interactive region filtering
- Plotly-based interactive visualizations
- Professional dashboard layout

## Tools and Technologies

- Python
- Pandas
- Plotly
- Dash
- Jupyter Notebook

## Dataset

The dashboard uses the `sales_data.csv` dataset containing sales transactions with information about:

- Date
- Product
- Quantity
- Price
- Customer ID
- Region
- Total Sales

In [ ]:
import pandas as pd
import plotly.express as px

from dash import Dash, dcc, html
from dash.dependencies import Input, Output

df = pd.read_csv("sales_data.csv")

df["Date"] = pd.to_datetime(df["Date"])


total_sales = df["Total_Sales"].sum()

average_sales = df["Total_Sales"].mean()

total_transactions = len(df)

top_product = (
    df.groupby("Product")["Total_Sales"]
    .sum()
    .idxmax()
)

top_region = (
    df.groupby("Region")["Total_Sales"]
    .sum()
    .idxmax()
)


app = Dash(__name__)

app.title = "Interactive Sales Dashboard"



app.layout = html.Div([

    # TITLE
    html.H1(
        "INTERACTIVE SALES DASHBOARD",
        style={
            "textAlign": "center",
            "color": "#1f2937",
            "fontSize": "36px",
            "marginBottom": "10px"
        }
    ),

    html.P(
        "Sales performance, product analysis and regional insights",
        style={
            "textAlign": "center",
            "color": "#6b7280",
            "fontSize": "18px",
            "marginBottom": "30px"
        }
    ),


    html.Div([

        html.Div([
            html.H4("TOTAL SALES"),
            html.H2(f"{total_sales:,.0f}")
        ], style={
            "backgroundColor": "#ffffff",
            "padding": "20px",
            "borderRadius": "10px",
            "textAlign": "center",
            "width": "18%"
        }),

        html.Div([
            html.H4("AVERAGE SALE"),
            html.H2(f"{average_sales:,.0f}")
        ], style={
            "backgroundColor": "#ffffff",
            "padding": "20px",
            "borderRadius": "10px",
            "textAlign": "center",
            "width": "18%"
        }),

        html.Div([
            html.H4("TRANSACTIONS"),
            html.H2(f"{total_transactions}")
        ], style={
            "backgroundColor": "#ffffff",
            "padding": "20px",
            "borderRadius": "10px",
            "textAlign": "center",
            "width": "18%"
        }),

        html.Div([
            html.H4("TOP PRODUCT"),
            html.H2(top_product)
        ], style={
            "backgroundColor": "#ffffff",
            "padding": "20px",
            "borderRadius": "10px",
            "textAlign": "center",
            "width": "18%"
        }),

        html.Div([
            html.H4("TOP REGION"),
            html.H2(top_region)
        ], style={
            "backgroundColor": "#ffffff",
            "padding": "20px",
            "borderRadius": "10px",
            "textAlign": "center",
            "width": "18%"
        })

    ], style={
        "display": "flex",
        "justifyContent": "space-between",
        "marginBottom": "30px"
    }),


    html.Div([

        html.Div([

            html.Label(
                "Select Product:",
                style={
                    "fontWeight": "bold",
                    "color": "#1f2937"
                }
            ),

            dcc.Dropdown(
                id="product-filter",
                options=[
                    {
                        "label": "All",
                        "value": "All"
                    }
                ] + [
                    {
                        "label": product,
                        "value": product
                    }
                    for product in sorted(df["Product"].unique())
                ],
                value="All",
                clearable=False
            )

        ], style={
            "width": "45%",
            "display": "inline-block",
            "marginRight": "5%"
        }),


        html.Div([

            html.Label(
                "Select Region:",
                style={
                    "fontWeight": "bold",
                    "color": "#1f2937"
                }
            ),

            dcc.Dropdown(
                id="region-filter",
                options=[
                    {
                        "label": "All",
                        "value": "All"
                    }
                ] + [
                    {
                        "label": region,
                        "value": region
                    }
                    for region in sorted(df["Region"].unique())
                ],
                value="All",
                clearable=False
            )

        ], style={
            "width": "45%",
            "display": "inline-block"
        })

    ], style={
        "marginBottom": "30px"
    }),


    dcc.Graph(
        id="sales-trend"
    ),

    html.Div([

        html.Div(
            dcc.Graph(
                id="product-chart"
            ),
            style={
                "width": "50%",
                "display": "inline-block"
            }
        ),

        html.Div(
            dcc.Graph(
                id="region-chart"
            ),
            style={
                "width": "50%",
                "display": "inline-block"
            }
        )

    ]),


    dcc.Graph(
        id="scatter-chart"
    )

], style={
    "backgroundColor": "#f3f4f6",
    "padding": "30px",
    "fontFamily": "Arial, sans-serif",
    "minHeight": "100vh"
})


@app.callback(
    Output("sales-trend", "figure"),
    Output("product-chart", "figure"),
    Output("region-chart", "figure"),
    Output("scatter-chart", "figure"),

    Input("product-filter", "value"),
    Input("region-filter", "value")
)
def update_dashboard(
    selected_product,
    selected_region
):

    filtered_df = df.copy()


    # Product filter
    if selected_product != "All":

        filtered_df = filtered_df[
            filtered_df["Product"] == selected_product
        ]


    # Region filter
    if selected_region != "All":

        filtered_df = filtered_df[
            filtered_df["Region"] == selected_region
        ]

    daily_sales = (
        filtered_df
        .groupby("Date")["Total_Sales"]
        .sum()
        .reset_index()
    )


    product_sales = (
        filtered_df
        .groupby("Product")["Total_Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Total_Sales",
            ascending=False
        )
    )


    region_sales = (
        filtered_df
        .groupby("Region")["Total_Sales"]
        .sum()
        .reset_index()
        .sort_values(
            "Total_Sales",
            ascending=False
        )
    )


    sales_trend = px.line(
        daily_sales,
        x="Date",
        y="Total_Sales",
        markers=True,
        title="Daily Sales Trend"
    )


    product_chart = px.bar(
        product_sales,
        x="Product",
        y="Total_Sales",
        title="Sales by Product"
    )


    region_chart = px.bar(
        region_sales,
        x="Region",
        y="Total_Sales",
        title="Sales by Region"
    )


    scatter_chart = px.scatter(
        filtered_df,
        x="Quantity",
        y="Total_Sales",
        color="Product",
        hover_data=[
            "Date",
            "Price",
            "Region"
        ],
        title="Quantity vs Total Sales"
    )


    return (
        sales_trend,
        product_chart,
        region_chart,
        scatter_chart
    )

In [4]:
app.run(
    jupyter_mode="external",
    debug=False,
    port=8050
)

Dash app running on http://127.0.0.1:8050/


# DASHBOARD OVERVIEW

## Key Performance Indicators

The dashboard provides five important KPIs:

1. **Total Sales** - Total revenue generated from all transactions.
2. **Average Sale** - Average sales value per transaction.
3. **Transactions** - Total number of sales transactions.
4. **Top Product** - Product generating the highest total sales.
5. **Top Region** - Region generating the highest total sales.

## Visualizations

### Daily Sales Trend

A line chart is used to visualize how sales change over time.

### Sales by Product

A bar chart compares total sales across different products.

### Sales by Region

A bar chart compares total sales across different geographical regions.

### Quantity vs Total Sales

A scatter plot shows the relationship between the quantity sold and total sales for individual transactions.

## Interactive Filters

Users can filter the dashboard using:

- Product selection
- Region selection

The charts update dynamically according to the selected filters.

# TECHNICAL IMPLEMENTATION

## Data Processing

Pandas is used to:

- Load the CSV dataset
- Convert the Date column into datetime format
- Group and aggregate sales data
- Calculate KPIs
- Filter data based on user selections

## Visualization

Plotly is used to create interactive charts with:

- Hover information
- Zooming
- Panning
- Interactive legends
- Dynamic chart updates

## Dashboard Framework

Dash is used to build the interactive dashboard interface.

The dashboard consists of:

- Header section
- KPI cards
- Filter controls
- Sales trend chart
- Product sales chart
- Regional sales chart
- Quantity versus sales scatter plot

## Callback Function

Dash callbacks are used to update the dashboard visualizations whenever the user changes the product or region filters.

# TESTING AND VALIDATION

The dashboard was tested to verify that all major components work correctly.

## Tests Performed

- Dataset loads successfully.
- Date values are processed correctly.
- KPI calculations are displayed correctly.
- Daily sales chart renders successfully.
- Product sales chart renders successfully.
- Regional sales chart renders successfully.
- Scatter plot renders successfully.
- Product filter works correctly.
- Region filter works correctly.
- Multiple filters can be combined.
- Plotly interactive features work correctly.
- Dashboard loads without callback errors.

## Result

All implemented dashboard components were successfully tested and displayed correctly.

# DAY 4 COMPLETED ✅

## Final Outcome

Day 4 successfully combined the visualization techniques developed throughout Week 6 into an interactive sales dashboard.

### Completed Features

- ✅ KPI cards
- ✅ Daily sales trend
- ✅ Product performance analysis
- ✅ Regional sales analysis
- ✅ Quantity versus sales analysis
- ✅ Interactive product filter
- ✅ Interactive region filter
- ✅ Plotly interactive charts
- ✅ Dash dashboard
- ✅ Dashboard testing and validation

## Week 6 Skills Demonstrated

- Data visualization
- Statistical visualization
- Interactive visualization
- Dashboard development
- Data filtering
- Data aggregation
- Pandas
- Seaborn
- Plotly
- Dash